In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from smmargins import Margins

rng = np.random.default_rng(7)
N = 5_000
df = pd.DataFrame({
    "age":    rng.normal(45, 12, N).clip(18, 90),
    "income": rng.lognormal(10.5, 0.4, N),
    "educ":   rng.choice(["hs", "college", "grad"], N, p=[0.4, 0.4, 0.2]),
    "female": rng.integers(0, 2, N),
})
eta = (-4.0 + 0.05 * df["age"] + 0.00001 * df["income"]
       + 0.8 * (df["educ"] == "college") + 1.4 * (df["educ"] == "grad")
       + 0.3 * df["female"] - 0.0004 * df["age"] * df["female"])
df["voted"] = (rng.uniform(0, 1, N) < 1 / (1 + np.exp(-eta))).astype(int)

fit = smf.logit("voted ~ age + income + C(educ) + female + age:female", data=df).fit(disp=False)

In [2]:
M = Margins(fit)

In [3]:
M.predict()

     prediction  std err          z  P>|z|  [95% Conf.  Interval]
AAP      0.3622  0.00634  57.128076    0.0    0.349774   0.374626

In [4]:
M.dydx("age")

         dy/dx   std err          z         P>|z|  [95% Conf.  Interval]
dage  0.010118  0.000505  20.037431  2.598390e-89    0.009128   0.011108

In [5]:
M.dydx("age").summary()

,dy/dx,std err,z,P>|z|,[95% Conf.,Interval]
dage,0.010118,0.000505,20.037431,2.598390e-89,0.009128,0.011108


In [6]:
M.dydx("female").summary()

,contrast,std err,z,P>|z|,[95% Conf.,Interval]
female: 1 vs 0,0.062486,0.012681,4.927392,8.333442e-07,0.037631,0.087341
